In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Download data
data = yf.download("AAPL", period="3y", interval="1d")

# Clean column headers if multi-index
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Indicators
data['SMA_20'] = data['Close'].rolling(window=20).mean()
data['SMA_50'] = data['Close'].rolling(window=50).mean()

# Generate Signals
data['Signal'] = 0
data.loc[data['SMA_20'] > data['SMA_50'], 'Signal'] = 1

# === Production Grade Risk Management Engine ===
initial_capital = 100000.0
cash = initial_capital
shares_owned = 0.0

# Pre-fill tracking columns
data['Position'] = 0.0
data['Liquid_Cash'] = initial_capital

# Get integer column positions to use ultra-fast .iat tool
pos_idx = data.columns.get_loc('Position')
Liquid_idx = data.columns.get_loc('Liquid_Cash')
signal_idx = data.columns.get_loc('Signal')
close_idx = data.columns.get_loc('Close')
open_idx = data.columns.get_loc('Open')

data.iat[0, Liquid_idx] = initial_capital
data.iat[1, Liquid_idx] = initial_capital

for i in range(2, len(data)):
    current_signal = data.iat[i-1, signal_idx]
    prev_signal = data.iat[i-2, signal_idx]
    new_price = data.iat[i, open_idx]
    
    # 1. Check for Buy Signal (Transition Cash -> Stock)
    if current_signal == 1 and prev_signal == 0:
        aval_cash = data.iat[i-1, Liquid_idx]  # Total liquid cash from previous day
        cash_to_spend = 0.5 * aval_cash  # Risk management: Only invest 50% of available capital
        
        shares_to_buy = cash_to_spend / new_price
        shares_owned += shares_to_buy
        cash -= cash_to_spend 
        
    # 2. Check for Sell Signal (Liquidate Stock -> Cash)
    elif current_signal == 0 and prev_signal == 1:
        cash += (shares_owned * new_price) 
        shares_owned = 0.0
        
    # Assign values directly to the single dataframe cell securely using .iat
    data.iat[i, pos_idx] = shares_owned
    data.iat[i, Liquid_idx] = cash

Total_Wealth = (data.iat[len(data)-1, pos_idx] * data.iat[len(data)-1, close_idx]) + data.iat[len(data)-1, Liquid_idx]

print("=== Advanced Backtest Results ===")
print(f"Final Portfolio Value : ${Total_Wealth:.2f}")
print(f"Total Return          : {(Total_Wealth / initial_capital - 1)*100:.2f}%")


[*********************100%***********************]  1 of 1 completed

=== Advanced Backtest Results ===
Final Portfolio Value : $115876.12
Total Return          : 15.88%
